# 03 — Diagnóstico, hotspots y escenarios

## Pregunta de negocio
¿Qué causas y combinaciones operativas concentran mayor impacto y cuáles son más accionables?

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.analysis import enrich, cause_pareto, delay_attribution_summary, hotspots, scenarios

In [2]:
df = pd.read_csv(ROOT / 'data/raw/ferry_operations_synthetic.csv', parse_dates=['scheduled_start','actual_start','scheduled_end','actual_end'])
df = enrich(df)

In [3]:
pareto = cause_pareto(df)
display(pareto)
ax = pareto.plot(x='disruption_reason', y='total_delay_min', kind='bar', figsize=(10,5), legend=False, title='Pareto de retraso atribuible')
ax.set_ylabel('Minutos'); plt.xticks(rotation=35); plt.tight_layout(); plt.show()

,disruption_reason,services,total_delay_min,avg_delay_min,p95_delay_min,total_delay_cost_eur,delay_share_pct,cumulative_share_pct,controllability_score
5,WEATHER,1628,23472.7,14.42,39.46,974153.87,36.63,36.63,0.25
4,TECHNICAL,1375,15627.4,11.37,31.13,699144.54,24.39,61.02,0.90
2,PORT_CONGESTION,1512,11059.6,7.31,18.55,515991.59,17.26,78.28,0.75
1,LATE_BOARDING,1004,5708.9,5.69,14.84,266353.90,8.91,87.19,0.85
0,CREW,619,5250.6,8.48,23.90,238352.96,8.19,95.38,0.90
3,SUPPLY_ISSUE,373,2957.5,7.93,21.28,132185.32,4.62,100.00,0.70


In [4]:
display(delay_attribution_summary(df))

,delay_attribution,services,total_delay_min,avg_delay_min,total_delay_cost_eur,delay_share_pct
0,ATTRIBUTED CAUSE,6511,64076.7,9.84,2826182.18,81.79
1,UNATTRIBUTED / NORMAL VARIABILITY,5375,14267.8,2.65,690388.85,18.21


In [5]:
hot = hotspots(df)
cols = ['route_id','hour','disruption_reason','services','p95_delay_min','otr_15_pct','total_delay_cost_eur','impact_index_0_100','actionability_index_0_100','priority']
display(hot[cols].head(15))

,route_id,hour,disruption_reason,services,p95_delay_min,otr_15_pct,total_delay_cost_eur,impact_index_0_100,actionability_index_0_100,priority
157,BCN-PMI,15,TECHNICAL,16,41.53,56.25,12258.40,54.58,51.31,MEDIUM
363,DEN-IBZ,16,TECHNICAL,26,29.50,69.23,12805.56,49.23,46.28,MEDIUM
170,BCN-PMI,17,WEATHER,19,60.59,63.16,19626.25,81.25,44.69,MEDIUM
470,DEN-PMI,17,WEATHER,17,65.30,35.29,15441.30,80.41,44.22,MEDIUM
70,BCN-IBZ,17,WEATHER,15,53.26,26.67,14109.70,75.78,41.68,MEDIUM
268,DEN-FOR,17,TECHNICAL,18,28.60,55.56,8831.83,40.18,37.77,LOW
547,MAH-BCN,13,WEATHER,15,74.85,60.00,13793.16,67.73,37.25,LOW
121,BCN-PMI,9,TECHNICAL,15,41.22,73.33,9714.31,38.90,36.57,LOW
663,VAL-IBZ,16,WEATHER,15,56.17,46.67,12165.33,66.34,36.49,LOW
152,BCN-PMI,14,WEATHER,21,39.20,57.14,14875.57,60.96,33.53,LOW


In [6]:
display(scenarios(df))

,scenario,assumption,delay_saved_min,delay_saved_pct,direct_delay_cost_saved_eur
2,WEATHER y TECHNICAL -10%,Reducción proporcional hipotética; no es una p...,3910.01,4.99,167329.84
0,WEATHER -10%,Reducción proporcional hipotética; no es una p...,2347.27,3.00,97415.39
4,PROPAGACIÓN DE ROTACIONES -20%,Reducción proporcional hipotética; no es una p...,1782.96,2.28,80029.94
1,TECHNICAL -10%,Reducción proporcional hipotética; no es una p...,1562.74,1.99,69914.45
3,PORT_CONGESTION -10%,Reducción proporcional hipotética; no es una p...,1105.96,1.41,51599.16


In [7]:
weather = df.loc[df.completed].groupby('weather').agg(servicios=('operation_id','count'), delay_medio=('delay_min','mean'), p95=('delay_min',lambda s:s.quantile(.95)), otr15=('on_time_15',lambda s:100*s.mean())).reset_index()
display(weather)
weather.plot(x='weather', y=['delay_medio','p95'], kind='bar', figsize=(9,5), title='Impacto de la meteorología')
plt.ylabel('Minutos'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

,weather,servicios,delay_medio,p95,otr15
0,CALM,6616,4.134689,13.20,96.100363
1,ROUGH,1728,11.459896,31.50,73.611111
2,STORM,438,24.268493,61.36,37.899543
3,WINDY,3104,6.622777,20.70,89.884021


## Conclusión

El Pareto separa causas atribuidas de variabilidad no atribuida. Los hotspots combinan impacto y controlabilidad, mientras que los escenarios deben interpretarse como sensibilidades y no como predicciones causales.